In [6]:
import pandas as pd 
import numpy as np 
import random
import uuid
from faker import Faker 
from datetime import datetime , timedelta

# data sample configuration 
num_customers = 100 
num_calls = 500000
start_date = datetime(2025,1,1)
end_date = datetime(2025,3,31)


rate_card = {
    ('Jordan' , 'Jordan') : {'price':0.005 , 'cost':0.02} , 
    ('Saudi Arabia' , 'Saudi Arabia') : {'price':0.08 , 'cost':0.04} , 
     ('UAE' , 'UAE') : {'price':0.10 , 'cost':0.06} , 
     ('Saudi Arabia' , 'Saudi Arabia') : {'price':0.08 , 'cost':0.04} , 

    ('Jordan', 'Saudi Arabia'):       {'price': 0.18, 'cost': 0.14},
    ('Jordan', 'UAE'):                {'price': 0.20, 'cost': 0.16},
    ('Jordan', 'Egypt'):              {'price': 0.15, 'cost': 0.12},
    
    # write a fix scinario for project 
    ('Jordan', 'Tunisia'):            {'price': 0.25, 'cost': 0.35}, 
    ('Saudi Arabia', 'Tunisia'):      {'price': 0.25, 'cost': 0.35},
    
}
# I will create a default_rate when the routes isn't appear in the rate card
default_rate = {'price':0.30 , 'cost':0.20} 


# I will use faker lib to make the customers 
fake = Faker()
Faker.seed(42)
np.random.seed(42)

customers = []
for i in range(1,num_customers + 1) : 
    cust_id = f"cus_{str(i).zfill(3)}"
    # set where the companies located randomly 
    hq = random.choice(['Jordan','Saudi Arabia' , 'UAE'])
    owned_dids = [hq]
    paid_seats = random.randint(10,50)
    # I will assume that active agents between 80 - 100 % 
    active_agents = int(paid_seats * random.uniform(0.8 , 1.0))

    # if the customer choose full seats plan (whatsapp and tel) then 50 just tele will be 45 (this is based on maqsam sitee )
    plan_price = random.choice([45,50])
    behavior = "normal"

    # other scenarios
    if i == 1 :
        comp_name , behavior , hq , owned_dids = "Tunisia Trders" , "trap" , "Jordan" , ["jordan"]
    elif i == 2 : 
        comp_name, behavior, paid_seats, active_agents = "Support Desk Co", "inbound_abuser", 5, 5
    elif i ==3 : 
        comp_name, behavior, hq, owned_dids = "Gulf Exporters", "missed_upsell", "Jordan", ["Jordan"]        
    elif i == 4 : 
        comp_name, behavior, paid_seats, active_agents = "Ghost Corp" , "zombie" , 100 , 12
    else : 
        comp_name = fake.company()
        
    customers.append({
        'customer_id': cust_id,
        'company_name': comp_name,
        'hq_country': hq,
        'owned_dids': "|".join(owned_dids),
        'paid_seats': paid_seats,
        'active_agents': active_agents,
        'seat_price': plan_price,
        'behavior': behavior
    })

df_customers = pd.DataFrame(customers)

# generating calls to make CDR 

time_diff = int((end_date - start_date).total_seconds())
random_seconds = np.random.randint(0,time_diff , num_calls)
timestamps = start_date + pd.to_timedelta(random_seconds , unit='s')


cust_weights = np.ones(num_customers)
# I will raise the wight of our scinarios customres to make them appear more in the calls dataset
cust_weights[0:4] = 5 
cust_ids_sampled = np.random.choice(df_customers['customer_id'] , size= num_calls , p=cust_weights/cust_weights.sum())

df_calls = pd.DataFrame ({
    'call_id':[str(uuid.uuid4()) for _ in range(num_calls)] , 
    'timestamp' : timestamps , 
    'customer_id' : cust_ids_sampled
})



df_calls = df_calls.merge(df_customers[['customer_id' , 'behavior' ,'hq_country' , 'owned_dids' , 'active_agents']] , on= 'customer_id') 


# I will set the rules now based on the "behavior" column 
def assign_call_attributes(row) : 
    b= row['behavior']
    hq = row ['hq_country']
    dids = row['owned_dids'].split('|')

    #set direction and distination 
    if b == 'trap' : 
        direction = 'outbound'
        dest = "Tunisia" 
    elif b == 'inbound_abuser' : 
        # I will make the propality of inbound more in this case cuze the scinario is inboud_abuser 
        direction = np.random.choice(["inbound", "outbound"], p=[0.95, 0.05])
        dest = hq if direction == "inbound" else "Saudi Arabia"
    elif b == 'missed_upsell' : 
        direction = 'outbound' 
        dest = 'Saudi Arabia'
    else : 
        direction = np.random.choice(["inbound", "outbound"], p=[0.3, 0.7])
        dest = np.random.choice([hq, "Saudi Arabia", "UAE", "Egypt", "UK"])

    origin = dest if dest in dids else hq 
    # i will use exponential distribution , to make the durations most sohrt and the other will be long 
    dur = round(np.random.exponential(scale = 4.0) , 2 )
    dur = max(0.1 , dur)

    agent_id = f"AGT_{row['customer_id']}_{np.random.randint(1,row['active_agents'] + 1 )}"

    return pd.Series([direction , origin , dest , dur , agent_id])

df_calls[['direction', 'caller_id_country', 'destination_country', 'duration_minutes', 'agent_id']] = df_calls.apply(assign_call_attributes, axis=1)

# I will add column for sip_code , to make the anomalies scinaro 
# the mapping is like this 200 --> healthy , 486 --> busy , 503 --> Error
df_calls['sip_code'] = np.random.choice([200,486,503] , size = num_calls , p = [0.9 , 0.08 , 0.02])


anomaly_mask = (
    (df_calls['destination_country'] =='Egypt') &
    (df_calls['timestamp'] >= '2025-02-15 10:00:00') & 
    (df_calls['timestamp'] <= '2025-02-15 14:00:00')
)

df_calls.loc[anomaly_mask , 'sip_code'] = 503
df_calls.loc[anomaly_mask , 'duration_minutes'] = 0 #--> this is faild calls so i will update the values to 0 


def calculate_rates(row):
    if row['direction'] == 'inbound' : 
        return pd.Series([0.00 , 0.01]) # selling price = 0 and cost = 1 cent 

    rate = rate_card.get((row['caller_id_country'],row['destination_country']) , default_rate)
    return pd.Series([rate['price'] , rate['cost']])

df_calls[['rate_price' , 'rate_cost']] = df_calls.apply(calculate_rates , axis= 1 )


# clean some coumns and sort 
df_calls = df_calls.drop(columns=['behavior', 'hq_country', 'owned_dids', 'active_agents'])
df_calls = df_calls.sort_values('timestamp').reset_index(drop=True)


# expoert to csv 

df_customers.to_parquet('dim_customers.parquet',index= False)
df_calls.to_parquet('fact_calls.parquet' , index= False)